In [1]:
from Load_Data import load_ptb_xl_data


path = "../../../../../data/ecg/db_projects/physionet/ptb_xl/raw/physionet.org/files/ptb-xl/1.0.3/"
train_data, test_data, val_data, train_id, test_id, val_id, train_leads, test_leads, val_leads = load_ptb_xl_data(path)
print(train_data.shape)

/pools/apollon/ummisco/data/projects/deepecg/analyses/17.gan/ECGenerator/GitHub/Scr/Load_Data.py:23: RuntimeWarning: invalid value encountered in divide
  return(-1+((Z-mini)*(2))/(maxi-mini))


(208904, 1, 5000)


In [2]:
from VQ_VAE import Encoder_net, Decoder_net, Quantize_net
import torch

size = 4
vocab_size = 64
experiment = 42
encoder = Encoder_net(size)
temp = torch.randn(1, 1, 5000)  # Dummy input to initialize the quantizer
temp = encoder(temp)  # Forward pass to initialize weights
encoding_size = temp.shape[1]  # Get the output size of the encoder
decoder = Decoder_net(size - 1, chanel_in=encoding_size)
quantizer = Quantize_net(encoding_size, vocab_size, experiment=experiment)
temp = temp.transpose(1, 2)  # Transpose to match the expected input shape for quantizer
quantized, commit_loss, embed_in, codebook = quantizer(temp)  # Forward pass to initialize weights
temp = quantized.transpose(1, 2)  # Transpose back to match the expected input shape for decoder
x_hat = decoder(temp)  # Forward pass to initialize weights


In [4]:
from Train_VQ_VAE import training, RMSE_Loss
from torch import optim

device = torch.device(1)

encoder = encoder.to(device)
decoder = decoder.to(device)
quantizer = quantizer.to(device)

optimizer_Encoder = optim.Adam(encoder.parameters(), lr=0.0001)
optimizer_Decoder = optim.Adam(decoder.parameters(), lr=0.0001)

loss = RMSE_Loss()
                
training(5, 128, train_data,test_data, Verbose = True, device = device, encoder = encoder, decoder = decoder, quantizer = quantizer, 
             optimizer_Encoder = optimizer_Encoder, optimizer_Decoder = optimizer_Decoder, 
             loss = loss, vocab_size = vocab_size, skip = 0, size = size, experiment=experiment)

KeyboardInterrupt: 